In [ ]:
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import defaultdict, Counter

import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

In [ ]:
# Load Dataset

df = pd.read_csv('/content/drive/MyDrive/Data Mining Lab/Search Engine Project/Air Conditioners.csv')

print("Dataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns)

print("\nFirst 10 rows:")
print(df.head(10))

**Select Required Columns**

In [ ]:
df = df[['name','main_category','sub_category']]

df = df.fillna('')

df['document'] = (
    df['name'] + " " +
    df['main_category'] + " " +
    df['sub_category']
)

print(df[['name','document']].head())

**Text Preprocessing**

In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess(text):

    text = text.lower()

    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    words = text.split()

    words = [w for w in words if w not in stop_words]

    return words

In [ ]:
print(preprocess("LG 1.5 Ton AI Dual Inverter Split AC"))

**Create Document IDs**

In [ ]:
df['doc_id'] = range(len(df))

print(df[['doc_id','name']].head())

**Build Inverted Index (CORE REQUIREMENT)**

In [ ]:
inverted_index = defaultdict(dict)

for _, row in df.iterrows():

    doc_id = row['doc_id']

    tokens = preprocess(row['document'])

    freq = Counter(tokens)

    for term, count in freq.items():

        inverted_index[term][doc_id] = count

In [ ]:
print(inverted_index['lg'])

**Document Frequency**

In [ ]:
document_frequency = {}

for term in inverted_index:

    document_frequency[term] = len(inverted_index[term])

**Calculate IDF**

In [ ]:
import math

N = len(df)

idf = {}

for term, dfreq in document_frequency.items():

    idf[term] = math.log(N / (1 + dfreq))

**TF-IDF Search Function**

In [ ]:
def tfidf_search(query):

    query_terms = preprocess(query)

    scores = defaultdict(float)

    for term in query_terms:

        if term in inverted_index:

            postings = inverted_index[term]

            for doc_id, tf in postings.items():

                scores[doc_id] += tf * idf[term]

    ranked_results = sorted(
        scores.items(),
        key=lambda x:x[1],
        reverse=True
    )

    return ranked_results

**Keyword Search**

In [25]:
results = tfidf_search("lg inverter")

for doc_id, score in results[:10]:

    print("\nTitle:", df.iloc[doc_id]['name'])
    print("Score:", round(score,3))


Title: LG 1.5 Ton 5 Star AI DUAL Inverter Split AC (Copper, Super Convertible 6-in-1 Cooling, HD Filter with Anti-Virus Protectio...
Score: 2.875

Title: LG 1 Ton 4 Star Ai Dual Inverter Split Ac (Copper, Super Convertible 6-In-1 Cooling, Hd Filter With Anti Virus Protection,...
Score: 2.875

Title: LG 1.5 Ton 3 Star AI DUAL Inverter Split AC (Copper, Super Convertible 6-in-1 Cooling, HD Filter with Anti-Virus Protectio...
Score: 2.875

Title: LG 1.5 Ton 2 Star DUAL Inverter Split AC (Copper, Convertible 4-in-1 Cooling, HD Filter with Anti-virus Protection, 2023 M...
Score: 2.875

Title: LG 1.5 Ton 4 Star AI DUAL Inverter Split AC (Copper, Super Convertible 6-in-1 Cooling, HD Filter with Anti-Virus Protectio...
Score: 2.875

Title: LG 1.0 Ton 5 Star DUAL Inverter Wi-Fi Window AC (Copper, Convertible 4-in-1 cooling, PW-Q12WUZA, 2022 Model, HD Filter, Wh...
Score: 2.875

Title: LG 0.8 Ton 3 Star AI DUAL Inverter Split AC (Copper, Super Convertible 6-in-1 Cooling, HD Filter with Anti Vir

**AND Query**

In [ ]:
def and_query(query):

    terms = preprocess(query)

    if not terms:
        return []

    result_docs = None

    for term in terms:

        if term not in inverted_index:
            return []

        docs = set(inverted_index[term].keys())

        if result_docs is None:
            result_docs = docs
        else:
            result_docs &= docs

    return list(result_docs)

In [31]:
and_query("lg inverter")

[1,
 2,
 3,
 263,
 264,
 265,
 395,
 271,
 400,
 146,
 274,
 21,
 22,
 405,
 661,
 28,
 160,
 416,
 35,
 548,
 165,
 173,
 305,
 436,
 310,
 184,
 314,
 188,
 317,
 68,
 74,
 76,
 209,
 82,
 214,
 215,
 219,
 354,
 101,
 357,
 231,
 233,
 361,
 363,
 364,
 622,
 241,
 242,
 370,
 244,
 245,
 118,
 247,
 377,
 122,
 635,
 636]

**OR Query**

In [32]:
def or_query(query):

    terms = preprocess(query)

    result_docs = set()

    for term in terms:

        if term in inverted_index:

            result_docs |= set(
                inverted_index[term].keys()
            )

    return list(result_docs)

In [33]:
or_query("lg samsung")

[512,
 1,
 2,
 3,
 513,
 514,
 515,
 263,
 264,
 265,
 516,
 269,
 271,
 15,
 274,
 21,
 22,
 25,
 27,
 28,
 287,
 31,
 35,
 548,
 547,
 40,
 298,
 305,
 50,
 310,
 56,
 314,
 60,
 317,
 67,
 68,
 74,
 331,
 76,
 81,
 82,
 346,
 95,
 353,
 354,
 101,
 357,
 359,
 361,
 618,
 363,
 364,
 620,
 622,
 621,
 366,
 370,
 118,
 377,
 121,
 122,
 635,
 636,
 395,
 400,
 146,
 405,
 661,
 671,
 160,
 416,
 420,
 165,
 424,
 682,
 683,
 173,
 430,
 688,
 436,
 692,
 184,
 440,
 441,
 188,
 446,
 453,
 458,
 463,
 209,
 214,
 215,
 219,
 479,
 481,
 482,
 231,
 233,
 241,
 242,
 244,
 245,
 247,
 253,
 510,
 511]

**Display Search Results**

In [34]:
def display_results(query):

    results = tfidf_search(query)

    print(f"\nSearch Query: {query}")
    print("="*50)

    for rank, (doc_id, score) in enumerate(results[:10], start=1):

        row = df.iloc[doc_id]

        print(f"\nRank #{rank}")

        print("Title:")
        print(row['name'])

        print("\nSnippet:")
        print(row['document'][:150])

        print("\nScore:")
        print(round(score,4))

        print("-"*50)

In [35]:
display_results("lg inverter")


Search Query: lg inverter

Rank #1
Title:
LG 1.5 Ton 5 Star AI DUAL Inverter Split AC (Copper, Super Convertible 6-in-1 Cooling, HD Filter with Anti-Virus Protectio...

Snippet:
LG 1.5 Ton 5 Star AI DUAL Inverter Split AC (Copper, Super Convertible 6-in-1 Cooling, HD Filter with Anti-Virus Protectio... appliances Air Condition

Score:
2.8753
--------------------------------------------------

Rank #2
Title:
LG 1 Ton 4 Star Ai Dual Inverter Split Ac (Copper, Super Convertible 6-In-1 Cooling, Hd Filter With Anti Virus Protection,...

Snippet:
LG 1 Ton 4 Star Ai Dual Inverter Split Ac (Copper, Super Convertible 6-In-1 Cooling, Hd Filter With Anti Virus Protection,... appliances Air Condition

Score:
2.8753
--------------------------------------------------

Rank #3
Title:
LG 1.5 Ton 3 Star AI DUAL Inverter Split AC (Copper, Super Convertible 6-in-1 Cooling, HD Filter with Anti-Virus Protectio...

Snippet:
LG 1.5 Ton 3 Star AI DUAL Inverter Split AC (Copper, Super Convertible 6-in-1 Cooli

**Interactive Search Engine**

In [26]:
while True:

    query = input("\nEnter Search Query: ")

    if query.lower() == "exit":
        break

    display_results(query)


Enter Search Query: samsung

Search Query: samsung

Rank #1
Title:
Samsung 1.5 Ton 3 Star Inverter Split AC (Copper, Convertible 5-in-1 Cooling Mode, Easy Filter Plus (Anti-Bacteria), 2023 ...

Snippet:
Samsung 1.5 Ton 3 Star Inverter Split AC (Copper, Convertible 5-in-1 Cooling Mode, Easy Filter Plus (Anti-Bacteria), 2023 ... appliances Air Condition

Score:
2.8416
--------------------------------------------------

Rank #2
Title:
Samsung 1.5 Ton 5 Star Inverter Split AC (Copper, Convertible 5-in-1 Cooling Mode, Anti-Bacteria, 2023 Model AR18CYNZABE W...

Snippet:
Samsung 1.5 Ton 5 Star Inverter Split AC (Copper, Convertible 5-in-1 Cooling Mode, Anti-Bacteria, 2023 Model AR18CYNZABE W... appliances Air Condition

Score:
2.8416
--------------------------------------------------

Rank #3
Title:
Samsung 1 Ton 3 Star Inverter Split Ac (Copper, Convertible 5-In-1 Cooling Mode, Easy Filter Plus (Anti-Bacteria), 2023 Mo...

Snippet:
Samsung 1 Ton 3 Star Inverter Split Ac (Copper, Convertibl

**GUI**

In [36]:
import gradio as gr

In [37]:
def search_engine(query):

    results = tfidf_search(query)

    output = ""

    for rank,(doc_id,score) in enumerate(results[:10],1):

        row = df.iloc[doc_id]

        output += f"""
Rank #{rank}

Title:
{row['name']}

Score:
{round(score,3)}

------------------------------------
"""

    return output

In [38]:
demo = gr.Interface(
    fn=search_engine,
    inputs="text",
    outputs="text",
    title="Amazon Air Conditioner Search Engine",
    description="Built using Inverted Index and TF-IDF Ranking"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://46caf7fa0ae396e019.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
